In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:
import random

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import euclidean_distances

import src.data_preprocessing.caption_generation as cg
import src.data_preprocessing.create_aux_data as cad
import src.data_preprocessing.data_utils as du
import src.data_preprocessing.gee_utils as gu
from src.data.base_caption_builder import BaseCaptionBuilder, DummyCaptionBuilder
from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

In [ ]:
BD = ButterflyDataset(
    modalities={"coords": None}, data_dir=os.path.join(os.environ["DATA_DIR"]), use_aux_data="all"
)

# BCB = ButterflyCaptionBuilder(
#     data_dir=os.path.join(os.environ["DATA_DIR"], "s2bms"),
#     templates_fname="v5.json",
#     concepts_fname="v4.json",
#     seed=42,
# )

# BM = BaseDataModule(
#     dataset=BD,
#     batch_size=32,
#     split_mode="from_file",
#     saved_split_file_name=os.path.join(
#         os.environ["DATA_DIR"], "s2bms", "splits/split_indices_s2bms_2024-08-14-1459.pth"
#     ),
#     caption_builder=BCB,
# )

In [ ]:
pth = "/Users/tplas/data/Countries_December_2025_Boundaries_UK_BFE_4990233815767821569/CTRY_DEC_2025_UK_BFE.shp"
gdf_uk = gpd.read_file(pth)
## merge multiple polygons into one (for the UK mainland)
gdf_uk = gdf_uk.dissolve()
gdf_uk = gdf_uk.to_crs(epsg=4326)

In [ ]:
pth = "../data/s2bms/splits/split_indices_s2bms_2024-08-14-1459.pth"
split_indices_original = torch.load(pth, weights_only=False)

split_names = ["train", "val", "test"]
gdf_splits = {}
for split_name in split_names:
    gdf_splits[split_name] = BD.df[
        np.isin(BD.df.name_loc, split_indices_original[f"{split_name}_indices"])
    ]
    gdf_splits[split_name] = gpd.GeoDataFrame(
        gdf_splits[split_name],
        geometry=gpd.points_from_xy(gdf_splits[split_name].lon, gdf_splits[split_name].lat),
        crs="EPSG:4326",
    )

gdf_s2bms = gpd.GeoDataFrame(
    BD.df,
    geometry=gpd.points_from_xy(BD.df.lon, BD.df.lat),
    crs="EPSG:4326",
)

fig, ax = plt.subplots(figsize=(10, 10))
gdf_uk.plot(ax=ax, color="lightgrey", edgecolor="black")
for split_name in split_names:
    gdf_splits[split_name].plot(ax=ax, label=split_name, markersize=20)
plt.legend()

In [ ]:
def average_dist_nearest_neighbour(n_samples):
    points_sampled = (
        gdf_uk.sample_points(n_samples, random_state=42).explode().reset_index(drop=True)
    )
    gdf_points_proj = points_sampled.to_crs("EPSG:27700")
    coords_points = np.array([(geom.x, geom.y) for geom in gdf_points_proj.geometry])

    # --- Compute distance matrix in km ---
    dist_matrix = euclidean_distances(coords_points, coords_points) / 1000
    np.fill_diagonal(dist_matrix, np.inf)
    nearest_distances = np.min(dist_matrix, axis=1)
    return nearest_distances.mean()


for n_samples in [1000, 2000, 5000, 10000, 20000]:
    avg_dist = average_dist_nearest_neighbour(n_samples)
    print(f"Average distance to nearest neighbour for {n_samples} samples: {avg_dist:.2f} km")

In [ ]:
n_samples = 10000
points_sampled = gdf_uk.sample_points(n_samples, random_state=42).explode().reset_index(drop=True)

gdf_s2bms_proj = gdf_s2bms.to_crs("EPSG:27700")  # British National Grid (UK data)
gdf_points_proj = points_sampled.to_crs("EPSG:27700")
coords_s2bms = np.array([(geom.x, geom.y) for geom in gdf_s2bms_proj.geometry])
coords_points = np.array([(geom.x, geom.y) for geom in gdf_points_proj.geometry])
dist_matrix = euclidean_distances(coords_s2bms, coords_points) / 1000

dist_df = pd.DataFrame(dist_matrix, index=gdf_s2bms_proj.index, columns=gdf_points_proj.index)

## Assign each sampled point to the data split of the closest S2BMS point, if the closest point is within a certain distance threshold (e.g., 5 km).
distance_threshold_km = 5
assigned_splits = []

for point_idx in dist_df.columns:
    closest_s2bms_idx = dist_df[point_idx].idxmin()
    closest_distance_km = dist_df.loc[closest_s2bms_idx, point_idx]

    if closest_distance_km <= distance_threshold_km:
        # Get the split of the closest S2BMS point
        split = None
        for split_name in split_names:
            if closest_s2bms_idx in gdf_splits[split_name].index:
                split = split_name
                break
        assigned_splits.append(split)
    else:
        assigned_splits.append(None)  # No close S2BMS point, so no split assigned

## Assign the remaining points to new splits, based on some centroid-based clustering:
inds_unassigned = [i for i, split in enumerate(assigned_splits) if split is None]
n_centroids = int(len(inds_unassigned) / 10)  # one centroid per 10 unassigned points
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=n_centroids, random_state=42)
kmeans.fit(coords_points[inds_unassigned])
centroid_labels = kmeans.labels_

target_ratios = {"train": 0.7, "val": 0.15, "test": 0.15}
centroid_to_split = {}
for centroid_label in np.unique(centroid_labels):
    inds_in_centroid = [
        inds_unassigned[i]
        for i in range(len(inds_unassigned))
        if centroid_labels[i] == centroid_label
    ]
    if len(inds_in_centroid) == 0:
        continue
    assigned_split = random.choices(
        population=list(target_ratios.keys()), weights=list(target_ratios.values()), k=1
    )[0]
    centroid_to_split[centroid_label] = assigned_split

for i, centroid_label in enumerate(centroid_labels):
    assert assigned_splits[inds_unassigned[i]] is None  # Only assign if not already assigned
    assigned_splits[inds_unassigned[i]] = centroid_to_split[centroid_label]

gdf_new_points = gpd.GeoDataFrame(
    {"split": assigned_splits}, geometry=points_sampled.geometry, crs="EPSG:4326"
)

from collections import Counter

print(Counter(assigned_splits))

In [ ]:
## Sample points randomly within gdf_uk

## convert from multipoint to geodataframe

## calculate distance between sampled points and points in gdf_splits:
# distances = {}
# for split_name in split_names:

colour_splits = {
    "train": "blue",
    "val": "orange",
    "test": "green",
}

fig, ax = plt.subplots(figsize=(10, 10))
gdf_uk.plot(ax=ax, color="lightgrey", edgecolor="black")
for split_name in split_names:
    gdf_new_points[gdf_new_points.split == split_name].plot(
        ax=ax,
        label=f"sampled {split_name}",
        markersize=5,
        alpha=0.5,
        color=colour_splits[split_name],
    )
    gdf_splits[split_name].plot(
        ax=ax, label=split_name, markersize=10, color=colour_splits[split_name], edgecolor="black"
    )
# points_sampled.plot(ax=ax, label="sampled points", color="red", markersize=5)
plt.legend()

In [ ]:
gdf_new_points["lon"] = gdf_new_points.geometry.x
gdf_new_points["lat"] = gdf_new_points.geometry.y
gdf_new_points["coords"] = list(zip(gdf_new_points.lon, gdf_new_points.lat))
gdf_new_points["name"] = [
    f"unlabelled-sample_{i}_{split}" for i, split in enumerate(gdf_new_points.split)
]
gdf_new_points

In [ ]:
save_file = False
if save_file:
    gdf_new_points.to_csv(
        os.path.join(os.environ["DATA_DIR"], "s2bms/source/", "unlabelled_samples_10k.csv"),
        index=False,
    )

In [ ]:
## Merge csv aux files:
save_file = False
folder_csv = os.path.join(os.environ["DATA_DIR"], "s2bms/source/")
assert os.path.exists(folder_csv), f"Folder {folder_csv} does not exist"
csv_files = [f for f in os.listdir(folder_csv) if f.endswith(".csv")]
aux_files = [f for f in csv_files if f.startswith("aux_data_unlabelled_samples")]

for ii, fn in enumerate(aux_files):
    df_aux = pd.read_csv(os.path.join(folder_csv, fn))
    if ii == 0:
        df_aux_all = df_aux
    else:
        df_aux_all = pd.concat([df_aux_all, df_aux], ignore_index=True)

df_samples = pd.read_csv(os.path.join(folder_csv, "unlabelled_samples_10k.csv"))

df_merge = pd.merge(df_samples, df_aux_all, on="name", how="left")
df_merge = df_merge.drop(columns=["coords_y", "coords_x", "geometry"])
dict_rename = {
    **{
        col: "aux_" + col for col in df_merge.columns if col not in ["name", "lon", "lat", "split"]
    },
    **{"name": "name_loc"},
}
df_merge = df_merge.rename(columns=dict_rename)

if save_file:
    df_merge.to_csv(
        os.path.join(folder_csv, "model_ready_s2bms-unlabelled-20260529.csv"), index=False
    )